In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import logging
from glob import glob
import random
import glob
from multiprocessing import Pool

In [3]:
imu_sensor_locations = ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand', 'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg',
                    'RightFoot', 'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe']

luo_sensor_locations = ['Pelvis', 'RightForeArm', 'RightUpperLeg', 'RightLowerLeg', 'LeftUpperLeg', 'LeftLowerLeg']

lower_body_sensor_locations = ['Pelvis', 'RightUpperLeg', 'RightLowerLeg', 'LeftUpperLeg', 'LeftLowerLeg', 'RightFoot', 'LeftFoot']

insole_sensor_locations = ['Arch', 'Hallux', 'Heel_L', 'Heel_R', 'Met1', 'Met3', 'Met5', 'Toes']

participant_num = [1, 2, 3, 4, 5, 7, 8, 10, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25]

##  insole df and imu df import & random segmentation selection

In [ ]:
# import base insole and base imu
insole_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/base_df_insole_df.csv')

In [ ]:
insole_df = insole_df[['time', 'participant_id', 'task', 'sensor_location',
                       'Left_norm', 'Left_raw', 'Right_norm', 'Right_raw',
                       'Left_norm_cumulative', 'Right_norm_cumulative']]

In [ ]:
insole_df['gait_cycle_id'] = insole_df['participant_id'].astype(str) + '_' + insole_df['task'].astype(str) + '_' + insole_df['time'].astype(str)

In [ ]:
imu_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/base_df_imu_df.csv')

In [ ]:
imu_df = imu_df[['time', 'participant_id', 'task', 'walk_mode', 'sensor_location',
                 'acceleration_x', 'acceleration_y', 'acceleration_z',
                 'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z']]

In [ ]:
insole_df['time'].equals(imu_df['time'])

In [ ]:
insole_df = insole_df[['time', 'participant_id', 'task', 'Left_norm', 'Right_norm', 'Left_norm_cumulative', 'Right_norm_cumulative', 'gait_cycle_id']]

In [ ]:
combo_df = pd.merge(imu_df, insole_df, on=['participant_id', 'task', 'time'], suffixes=('_imu', '_insole'), how='outer')

In [ ]:
del insole_df
del imu_df

In [ ]:
combo_df.columns

Index(['time', 'participant_id', 'task', 'walk_mode', 'sensor_location',
       'acceleration_x', 'acceleration_y', 'acceleration_z',
       'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z',
       'Left_norm', 'Right_norm', 'Left_norm_cumulative',
       'Right_norm_cumulative', 'gait_cycle_id'],
      dtype='object')

In [ ]:
combo_no_dupes = combo_df.drop_duplicates(subset=['time', 'participant_id', 'task'])

In [ ]:
len(combo_no_dupes)

2026204

In [ ]:
del combo_df

In [ ]:
def sliding_window(data, window_size, step_size):
    num_segments = (len(data) - window_size) // step_size + 1
    indices = np.arange(0, num_segments * step_size, step_size)
    return [data[i:i + window_size] for i in indices]

segment_size = 85
overlap = 0.75
step_size = int(segment_size * (1 - overlap))
print(f"Step size: {step_size}")
participants = combo_no_dupes['participant_id'].unique()
tasks = combo_no_dupes['task'].unique()

for p in participants:
    print(f"Processing participant: {p}")
    participant_df = combo_no_dupes[combo_no_dupes['participant_id'] == p]
    participant_segments = []

    for t in tasks:
        print(f"Processing task: {t}")
        task_df = participant_df[participant_df['task'] == t]

        unique_times = task_df['time'].unique()
        if len(unique_times) < segment_size:
            continue  # Skip if not enough data for a segment

        count = 0
        for segment in sliding_window(unique_times, segment_size, step_size):
            count += 1
            segment_mask = task_df['time'].isin(segment)
            segment_df = task_df.loc[segment_mask]
            segment_df['gait_cycle_id'] = f'{p}_{t}_{count}'
            segment_df['segment_count'] = count

            # Check for uniform surface type
            if segment_df['walk_mode'].nunique() == 1:
                min_time, max_time = segment_df['time'].min(), segment_df['time'].max()
                participant_segments.append(segment_df)


    # concat
    participant_segments_df = pd.concat(participant_segments, ignore_index=True)
    participant_segments_df.to_csv(f'/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg/ppant_{p}_segments.csv', index=False)

print("Segmentation complete.")


Output hidden; open in https://colab.research.google.com to view.

## reupload of randomly selected segments (to save RAM)

In [4]:
segmented_dfs = []

# navigate to folder and upload csvs in folder
os.chdir('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg')

for file in glob.glob("*.csv"):
    # import to df
    df = pd.read_csv(file)
    segmented_dfs.append(df)



In [5]:
# segmented_dfs[1]

In [6]:
for df in range(len(segmented_dfs)):
    # keep only specific columns
    segmented_dfs[df] = segmented_dfs[df][['time', 'participant_id', 'task', 'walk_mode', 'gait_cycle_id', 'segment_count']]


In [7]:
segmented_dfs[0]

,time,participant_id,task,walk_mode,gait_cycle_id,segment_count
0,0,1,A,walk,1_A_1,1
1,16,1,A,walk,1_A_1,1
2,33,1,A,walk,1_A_1,1
3,50,1,A,walk,1_A_1,1
4,66,1,A,walk,1_A_1,1
...,...,...,...,...,...,...
371190,404533,1,C,walk,1_C_1153,1153
371191,404550,1,C,walk,1_C_1153,1153
371192,404566,1,C,walk,1_C_1153,1153
371193,404583,1,C,walk,1_C_1153,1153


In [8]:
alt_seg_df = pd.concat(segmented_dfs, ignore_index=True)

## insole + segmentations

In [ ]:
# import base insole and base imu
insole_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/base_df_insole_df.csv')

In [ ]:
insole_df = insole_df[['time', 'participant_id', 'task', 'sensor_location', 'walk_mode', 'Left_norm', 'Right_norm', 'Left_norm_cumulative', 'Right_norm_cumulative']]

In [ ]:
# merge alt_seg left onto insole_df
combo_df = pd.merge(insole_df, alt_seg_df, on=['participant_id', 'task', 'time'], suffixes=('_insole', '_alt_seg'), how='left')

In [ ]:
# combo_df.head()

In [ ]:
# combo_df['gait_cycle_id'].value_counts()

In [ ]:
# export to csv
combo_df.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg_insole_combo_df.csv', index=False)

KeyboardInterrupt: 

In [33]:
del combo_df

## imu + segmentations

In [34]:
imu_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/base_df_imu_df.csv')

In [35]:
imu_df = imu_df[['time', 'participant_id', 'task', 'walk_mode', 'sensor_location',
                 'acceleration_x', 'acceleration_y', 'acceleration_z',
                 'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z']]

In [36]:
# merge alt_seg left onto insole_df
combo_df = pd.merge(imu_df, alt_seg_df, on=['participant_id', 'task', 'time'], suffixes=('_insole', '_alt_seg'), how='left')

In [37]:
# combo_df['sensor_location'].value_counts()

In [38]:
# combo_df['gait_cycle_id'].value_counts()

In [39]:
# export to csv
combo_df.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg_imu_combo_df.csv', index=False)

In [40]:
del combo_df

## insole reupload segmented before interpolation

In [ ]:
# label encoding the task column
from sklearn.preprocessing import LabelEncoder

In [ ]:
# import base insole
insole_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg_insole_combo_df.csv')

In [ ]:
insole_df.head(250)

,time,participant_id,task,sensor_location,walk_mode_insole,Left_norm,Right_norm,Left_norm_cumulative,Right_norm_cumulative,walk_mode_alt_seg,gait_cycle_id,segment_count
0,0,1,0,Arch,walk,0.001299,0.160105,0.838253,3.509033,walk,1_A_1,1.0
1,0,1,0,Hallux,walk,0.000223,0.968475,0.838253,3.509033,walk,1_A_1,1.0
2,0,1,0,Heel_L,walk,0.457522,0.000275,0.838253,3.509033,walk,1_A_1,1.0
3,0,1,0,Heel_R,walk,0.372672,0.002728,0.838253,3.509033,walk,1_A_1,1.0
4,0,1,0,Met1,walk,0.001484,0.987995,0.838253,3.509033,walk,1_A_1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
245,416,1,0,Met5,walk,0.674458,0.002136,2.494302,0.227988,walk,1_A_2,2.0
246,416,1,0,Toes,walk,0.060901,0.000050,2.494302,0.227988,walk,1_A_1,1.0
247,416,1,0,Toes,walk,0.060901,0.000050,2.494302,0.227988,walk,1_A_2,2.0
248,433,1,0,Arch,walk,1.000000,0.060116,2.551036,0.227942,walk,1_A_1,1.0


In [ ]:
# make letter column numeric
le = LabelEncoder()
insole_df['task'] = le.fit_transform(insole_df['task'])
insole_df['walk_mode'] = le.fit_transform(insole_df['walk_mode_insole'])

In [ ]:
insole_df[['walk_mode', 'walk_mode_insole']].value_counts()

,,count
walk_mode,walk_mode_insole,
4,walk,26790150
0,slope_down,8749336
1,slope_up,6485912
3,stairs_up,3772832
2,stairs_down,2151480


In [ ]:
insole_df.sort_values(by=['participant_id', 'task', 'segment_count', 'time'])

,time,participant_id,task,sensor_location,walk_mode_insole,Left_norm,Right_norm,Left_norm_cumulative,Right_norm_cumulative,walk_mode_alt_seg,gait_cycle_id,segment_count,walk_mode
0,0,1,0,Arch,walk,0.001299,0.160105,0.838253,3.509033,walk,1_A_1,1.0,4
1,0,1,0,Hallux,walk,0.000223,0.968475,0.838253,3.509033,walk,1_A_1,1.0,4
2,0,1,0,Heel_L,walk,0.457522,0.000275,0.838253,3.509033,walk,1_A_1,1.0,4
3,0,1,0,Heel_R,walk,0.372672,0.002728,0.838253,3.509033,walk,1_A_1,1.0,4
4,0,1,0,Met1,walk,0.001484,0.987995,0.838253,3.509033,walk,1_A_1,1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47940195,147684,22,0,Heel_R,walk,0.760375,0.001182,1.598241,2.063734,NaN,NaN,NaN,4
47940196,147684,22,0,Met1,walk,0.000223,0.269436,1.598241,2.063734,NaN,NaN,NaN,4
47940197,147684,22,0,Met3,walk,0.000000,0.226266,1.598241,2.063734,NaN,NaN,NaN,4
47940198,147684,22,0,Met5,walk,0.000584,0.440242,1.598241,2.063734,NaN,NaN,NaN,4


## imu reupload segmented before interpolation

In [9]:
# import base imu
imu_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg_imu_combo_df.csv')

In [10]:
imu_df.head()

,time,participant_id,task,walk_mode_insole,sensor_location,acceleration_x,acceleration_y,acceleration_z,angularVelocity_x,angularVelocity_y,angularVelocity_z,walk_mode_alt_seg,gait_cycle_id,segment_count
0,0,1,A,walk,LeftFoot,-2.857478,-0.923466,-1.930802,-0.401183,0.804390,0.147177,walk,1_A_1,1.0
1,0,1,A,walk,LeftLowerLeg,-9.046621,2.399089,-4.681049,0.408786,1.891735,0.834543,walk,1_A_1,1.0
2,0,1,A,walk,LeftUpperLeg,13.051041,3.724914,7.346136,0.765089,-0.045795,-0.434453,walk,1_A_1,1.0
3,0,1,A,walk,Pelvis,-3.558701,-1.303026,-0.356919,0.295869,-0.253239,1.032760,walk,1_A_1,1.0
4,0,1,A,walk,RightFoot,8.593493,-7.461521,3.174677,2.996236,8.846181,0.824499,walk,1_A_1,1.0


In [11]:
# label encoding the task column
from sklearn.preprocessing import LabelEncoder

# make letter column numeric
le = LabelEncoder()
imu_df['task'] = le.fit_transform(imu_df['task'])
imu_df['walk_mode'] = le.fit_transform(imu_df['walk_mode_insole'])

In [12]:
imu_df.sort_values(by=['participant_id', 'task', 'segment_count', 'time'])

,time,participant_id,task,walk_mode_insole,sensor_location,acceleration_x,acceleration_y,acceleration_z,angularVelocity_x,angularVelocity_y,angularVelocity_z,walk_mode_alt_seg,gait_cycle_id,segment_count,walk_mode
0,0,1,0,walk,LeftFoot,-2.857478,-0.923466,-1.930802,-0.401183,0.804390,0.147177,walk,1_A_1,1.0,4
1,0,1,0,walk,LeftLowerLeg,-9.046621,2.399089,-4.681049,0.408786,1.891735,0.834543,walk,1_A_1,1.0,4
2,0,1,0,walk,LeftUpperLeg,13.051041,3.724914,7.346136,0.765089,-0.045795,-0.434453,walk,1_A_1,1.0,4
3,0,1,0,walk,Pelvis,-3.558701,-1.303026,-0.356919,0.295869,-0.253239,1.032760,walk,1_A_1,1.0,4
4,0,1,0,walk,RightFoot,8.593493,-7.461521,3.174677,2.996236,8.846181,0.824499,walk,1_A_1,1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60186331,399517,25,2,walk,Pelvis,-1.123330,-5.000308,0.799571,-0.521903,-0.969002,-0.136810,NaN,NaN,NaN,4
60186332,399517,25,2,walk,RightFoot,0.817734,-2.902324,-0.960745,-1.065136,0.298337,0.006019,NaN,NaN,NaN,4
60186333,399517,25,2,walk,RightForeArm,-1.204089,1.700301,2.783419,1.204641,-0.878843,1.432499,NaN,NaN,NaN,4
60186334,399517,25,2,walk,RightLowerLeg,-9.409014,-6.376233,-3.877882,-1.218898,1.553173,0.407313,NaN,NaN,NaN,4


## interpolation

In [13]:
def interpolate_to_fixed_length(group, target_length=80):
    interpolated_group = []

    # Iterate over each sensor location within the group
    for sensor_location, data in group.groupby('sensor_location'):
        # Select only numeric columns for interpolation
        numeric_data = data.select_dtypes(include=[np.number])
        numeric_data = numeric_data.reset_index(drop=True)
        original_length = len(numeric_data)

        if original_length < 2:
            # Skip groups with insufficient data for interpolation
            print(f"Skipping sensor_location '{sensor_location}' with insufficient data.")
            continue

        # Create new evenly spaced indices
        new_indices = np.linspace(0, original_length - 1, target_length)

        # Interpolate the numeric data to the new indices
        interpolated_data = pd.DataFrame(
            {col: np.interp(new_indices, np.arange(original_length), numeric_data[col])
             for col in numeric_data.columns},
            index=new_indices
        )

        # Add back the sensor location information
        interpolated_data['sensor_location'] = sensor_location
        interpolated_data['walk_mode'] = group['walk_mode'].iloc[0]

        # Append to the results list
        interpolated_group.append(interpolated_data)

    # Combine all sensor locations back into a single DataFrame
    return pd.concat(interpolated_group, axis=0) if interpolated_group else pd.DataFrame()



def process_gait_cycle(group):
    return interpolate_to_fixed_length(group, target_length=80)


In [14]:
# Define the interpolation function
def interpolate_group(df, target_length=80):
    # Use your custom interpolation logic
    # Assuming 'time' is the x-axis and other columns are y-values
    df = df.sort_values(by='time')  # Ensure sorting
    interpolated = interpolate_to_fixed_length(df, target_length=target_length)
    return interpolated

# Group the DataFrame by 'participant_id' and 'gait_cycle_id'
grouped = imu_df.groupby(['participant_id', 'gait_cycle_id'])

# Apply the interpolation to each group
interpolated_results = grouped.apply(lambda group: interpolate_group(group))

# Reset index if needed
interpolated_results = interpolated_results.reset_index(drop=True)


<ipython-input-14-23584ea93368>:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  interpolated_results = grouped.apply(lambda group: interpolate_group(group))


In [13]:
# # Group the DataFrame by 'participant_id' and 'gait_cycle_id'
# grouped = insole_df.groupby(['participant_id', 'gait_cycle_id'])

# # Apply the interpolation to each group
# interpolated_results = grouped.apply(lambda group: interpolate_group(group))

# # Reset index if needed
# interpolated_results = interpolated_results.reset_index(drop=True)


In [15]:
participant_1 = interpolated_results[interpolated_results['participant_id'] == 1]

In [16]:
participant_1

,time,participant_id,task,acceleration_x,acceleration_y,acceleration_z,angularVelocity_x,angularVelocity_y,angularVelocity_z,segment_count,walk_mode,sensor_location
0,0.000000,1.0,0.0,-2.857478,-0.923466,-1.930802,-0.401183,0.804390,0.147177,1.0,4,LeftFoot
1,17.075949,1.0,0.0,-1.847266,-1.652109,0.475515,-0.224245,0.323846,-0.075986,1.0,4,LeftFoot
2,35.151899,1.0,0.0,-0.700074,1.813367,-0.616068,-0.233150,0.085054,-0.044008,1.0,4,LeftFoot
3,53.037975,1.0,0.0,-0.530070,-0.230923,0.220474,-0.254222,0.000632,-0.055738,1.0,4,LeftFoot
4,70.303797,1.0,0.0,-0.363445,0.122458,0.235417,-0.164618,-0.042471,-0.029016,1.0,4,LeftFoot
...,...,...,...,...,...,...,...,...,...,...,...,...
2794875,350628.696203,1.0,2.0,-2.209623,1.851872,-1.087140,1.315030,1.796499,1.931248,999.0,4,RightUpperLeg
2794876,350646.772152,1.0,2.0,-1.582954,1.279967,-1.132086,1.503826,1.738332,1.829802,999.0,4,RightUpperLeg
2794877,350663.974684,1.0,2.0,0.549650,-0.056238,-1.595381,1.826055,1.731965,1.753060,999.0,4,RightUpperLeg
2794878,350681.924051,1.0,2.0,0.162985,-1.712866,-1.797457,2.255534,1.801685,1.390736,999.0,4,RightUpperLeg


In [17]:
participant_1[['participant_id', 'task', 'segment_count']].value_counts()

participant_id  task  segment_count
1.0             0.0   1.0              640
                1.0   1198.0           640
                      1208.0           640
                      1207.0           640
                      1206.0           640
                                      ... 
                0.0   1614.0           640
                      1615.0           640
                      1616.0           640
                      1617.0           640
                2.0   1153.0           640
Name: count, Length: 4367, dtype: int64

In [18]:
interpolated_results.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/interpolated_alt_seg_imu_sensor_df.csv', index=False)

In [ ]:
# interpolated_results.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/interpolated_alt_seg_insole_sensor_df.csv', index=False)